# Exploracion del dataset Olist

**Fase 0 - Fundamentos y entorno de trabajo.**

Este notebook es exploratorio: sirve para entender el modelo relacional de
Olist (grano de cada tabla, claves primarias/foraneas, calidad de los
timestamps) antes de disenar los contratos de datos en la Fase 1. **No es
parte del pipeline final** — ninguna funcion de aqui se reutiliza en
`src/`.

Dataset: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

RAW = Path("../data/raw")
sorted(p.name for p in RAW.glob("*.csv"))

## 1. Carga y esquema de cada tabla

Cargamos las 9 tablas con Polars (lazy scan + `collect`, para acostumbrarnos
al patron que vamos a usar en Bronze) y revisamos shape + dtypes inferidos.

In [ ]:
tables = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

dfs: dict[str, pl.DataFrame] = {
    name: pl.scan_csv(RAW / fname).collect() for name, fname in tables.items()
}

# product_category_name_translation.csv trae BOM en el header
dfs["category_translation"].columns = [
    c.lstrip("\ufeff") for c in dfs["category_translation"].columns
]

for name, df in dfs.items():
    print(f"{name:22s} shape={df.shape}")

In [ ]:
for name, df in dfs.items():
    print(f"--- {name} ---")
    print(df.schema)
    print()

## 2. Grano de cada tabla (unicidad de la clave primaria)

`n_rows == n_unique(key)` confirma que la columna (o combinacion de
columnas) propuesta es realmente la PK de la tabla.

In [ ]:
candidate_keys = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"],
}

for name, key in candidate_keys.items():
    df = dfs[name]
    n_unique = df.select(key).n_unique()
    status = "OK" if n_unique == df.height else "** NO es PK simple **"
    print(f"{name:22s} key={key!s:35s} n_rows={df.height:7d} n_unique={n_unique:7d}  {status}")

`order_reviews` es un caso conocido: `review_id` puede repetirse cuando un
cliente deja mas de una resena para el mismo pedido. Si el resultado arriba
marca "NO es PK simple", queda anotado como algo a resolver en el diseno de
Silver (Fase 1/3): ¿la PK real es `review_id`, o es `(order_id, review_id)`,
o hay que quedarse con la ultima resena por pedido?

## 3. Relaciones (FKs) y registros huerfanos

Para cada FK candidata, contamos cuantos valores del lado "muchos" no
existen del lado "uno". Si el pipeline asume integridad referencial estricta
en Silver/Gold (fail-fast, ADR 0001), necesitamos saber de antemano si la
fuente ya trae huerfanos.

In [ ]:
fk_checks = [
    (
        "orders.customer_id",
        "customers.customer_id",
        "orders",
        "customer_id",
        "customers",
        "customer_id",
    ),
    ("order_items.order_id", "orders.order_id", "order_items", "order_id", "orders", "order_id"),
    (
        "order_items.product_id",
        "products.product_id",
        "order_items",
        "product_id",
        "products",
        "product_id",
    ),
    (
        "order_items.seller_id",
        "sellers.seller_id",
        "order_items",
        "seller_id",
        "sellers",
        "seller_id",
    ),
    (
        "order_payments.order_id",
        "orders.order_id",
        "order_payments",
        "order_id",
        "orders",
        "order_id",
    ),
    (
        "order_reviews.order_id",
        "orders.order_id",
        "order_reviews",
        "order_id",
        "orders",
        "order_id",
    ),
    (
        "products.product_category_name",
        "category_translation.product_category_name",
        "products",
        "product_category_name",
        "category_translation",
        "product_category_name",
    ),
]

for label_from, label_to, tbl_from, col_from, tbl_to, col_to in fk_checks:
    left = dfs[tbl_from].select(col_from).drop_nulls().unique()
    right = dfs[tbl_to].select(pl.col(col_to).alias(col_from)).drop_nulls().unique()
    orphans = left.join(right, on=col_from, how="anti")
    print(f"{label_from:38s} -> {label_to:42s} huerfanos={orphans.height}")

## 4. Distribucion de `order_status`

Relevante para el diseno de fail-fast y del SCD2 de estados de pedido
(Fase 1): que estados existen realmente y con que frecuencia.

In [ ]:
status_counts = (
    dfs["orders"].group_by("order_status").agg(pl.len().alias("n")).sort("n", descending=True)
)
status_counts

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(status_counts["order_status"].to_list(), status_counts["n"].to_list())
plt.xticks(rotation=45, ha="right")
plt.ylabel("n pedidos")
plt.title("Distribucion de order_status")
plt.tight_layout()
plt.show()

## 5. Timestamps de `orders` — ¿cual ancla el "dia de llegada simulada"?

`orders` trae 5 timestamps candidatos: `order_purchase_timestamp`,
`order_approved_at`, `order_delivered_carrier_date`,
`order_delivered_customer_date`, `order_estimated_delivery_date`.

Esta es la pregunta abierta que el plan deja para la Fase 1. Antes de
decidir, miramos: rango de fechas cubierto, y tasa de nulos por columna
(¿los nulos se explican por el `order_status`?).

In [ ]:
ts_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders_ts = dfs["orders"].with_columns(
    [pl.col(c).str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False) for c in ts_cols]
)

n = orders_ts.height
for c in ts_cols:
    n_null = orders_ts.select(pl.col(c).is_null().sum()).item()
    lo = orders_ts.select(pl.col(c).min()).item()
    hi = orders_ts.select(pl.col(c).max()).item()
    print(f"{c:32s} nulos={n_null:6d} ({n_null / n:5.1%})   rango=[{lo} .. {hi}]")

In [ ]:
# Nulos por timestamp, desglosados por order_status: ¿un timestamp nulo
# siempre corresponde a un estado que aun no llego a esa etapa?
(
    orders_ts.group_by("order_status")
    .agg(
        pl.len().alias("n"),
        *[pl.col(c).is_null().sum().alias(f"{c}_nulos") for c in ts_cols],
    )
    .sort("n", descending=True)
)

## 6. Geolocalizacion: `zip_code_prefix` no es una FK limpia

`geolocation` trae multiples lat/lng por `geolocation_zip_code_prefix`
(varios puntos GPS reportados dentro del mismo prefijo postal). No hay una
fila unica por prefijo — cualquier join contra `customers`/`sellers` por
zip prefix va a multiplicar filas si no se agrega antes.

In [ ]:
geo = dfs["geolocation"]
dup = (
    geo.group_by("geolocation_zip_code_prefix")
    .agg(pl.len().alias("n_puntos"))
    .sort("n_puntos", descending=True)
)
print(f"zip prefixes unicos: {dup.height}  |  filas totales: {geo.height}")
dup.head(10)

## 7. Categorias de producto

Cobertura de la tabla de traduccion, y cuantos productos quedan sin
categoria (`product_category_name` nulo).

In [ ]:
n_null_cat = dfs["products"].select(pl.col("product_category_name").is_null().sum()).item()
print(f"productos sin categoria: {n_null_cat} / {dfs['products'].height}")

cats_productos = dfs["products"].select("product_category_name").drop_nulls().unique()
cats_traduccion = dfs["category_translation"].select("product_category_name").unique()
sin_traduccion = cats_productos.join(cats_traduccion, on="product_category_name", how="anti")
print(f"categorias en products sin fila en category_translation: {sin_traduccion.height}")
sin_traduccion

## 8. Resumen para discutir en Fase 1

Preguntas que este notebook deja planteadas con datos reales (no solo
documentacion de Kaggle) para cerrar el diseno de contratos:

- ¿Que timestamp ancla el "dia de llegada simulada" de un pedido —
  `order_purchase_timestamp`? ¿Que pasa con pedidos cuyo
  `order_delivered_customer_date` cae dias/semanas despues — generan un
  nuevo registro en Bronze ese dia, o se hace upsert del estado?
- ¿`order_reviews` tiene `review_id` duplicado? Si si, ¿cual es la PK real
  para el modelo Silver/Gold?
- ¿Hay huerfanos reales en las FKs revisadas en la seccion 3? Si los hay,
  el fail-fast total (ADR 0001) los va a tumbar apenas entren a Silver —
  vale la pena saberlo antes de correr el pipeline por primera vez.
- `geolocation` requiere agregacion (ej. promedio de lat/lng por zip
  prefix) antes de poder usarse como dimension limpia en Gold.